# Fraud Compliance Agent Notebook 04 — Feature availability matrix

**Fraud Compliance Agent · P0-03/P0-04 — feature eligibility decision**  
**Status:** Proposal prepared — review pending  
**CRISP-DM phases:** Business understanding → Data understanding → Data preparation → Evaluation → Deployment  
**Decision supported:** P0-03/P0-04 — feature eligibility decision

---

## In plain English

This notebook is a **truth table for possible fraud signals**. It asks, for each idea we might want to use — such as transaction amount, account history, or payment direction — whether our data can genuinely provide it at the moment a decision would be made.

It is like making an ingredients list before choosing a recipe: a feature can be used only if the ingredient is actually available, in time, and in a reliable form. This notebook does not create new data, train a model, or approve a feature for production. Missing information stays clearly labelled as unavailable or simulated.

Read the steps as: list candidate signals, compare each data source, check timing, then publish the exclusions a reviewer needs to see.

<a id="purpose"></a>
## Purpose

**Question:** Which proposed features are observed, derivable, unavailable, or simulated across each source and scenario?

Prevent future models or screens from claiming signals that the data cannot supply at decision time.

### Non-goals

- No production ingestion, policy, contract, or model release is approved by this notebook.
- No raw provider payloads, identifiers, credentials, customer data, model weights, or hidden reasoning may enter Git.
- Missing evidence remains unknown; it must never become a fabricated value or result.

<a id="contents"></a>
## Contents

1. [Purpose](#purpose)
2. [Pre-flight and safety](#pre-flight)
3. [Define candidate features](#step-1)
4. [Compare source availability](#step-2)
5. [Check point-in-time eligibility](#step-3)
6. [Publish exclusions](#step-4)
7. [Findings, limitations, and next gate](#review)

---

<a id="pre-flight"></a>
## Pre-flight and safety

Run from the repository with the **Fraud Compliance Agent API (Python 3.13)** kernel. List environment variables by name only. Clear every output before committing.

**Expected sanitised artifact:** `docs/proposals/feature-availability.proposed.json`


In [ ]:
from __future__ import annotations

import json
import subprocess
from datetime import UTC, datetime
from pathlib import Path

REPOSITORY_ROOT = Path.cwd().resolve()
while REPOSITORY_ROOT != REPOSITORY_ROOT.parent and not (REPOSITORY_ROOT / "docs" / "project-context.md").exists():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent

if not (REPOSITORY_ROOT / "docs" / "project-context.md").exists():
    raise RuntimeError("Run this notebook from inside the fraud-compliance-agent repository.")

def git_revision() -> str:
    """Return the current Git revision without failing a proposal-only run.

    Returns:
        Commit hash or an explicit uncommitted-or-unavailable marker.

    Side effects:
        Runs a read-only Git command; no data or secrets are accessed.
    """
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=REPOSITORY_ROOT, text=True, stderr=subprocess.DEVNULL
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        return "uncommitted-or-unavailable"

RUN_CONTEXT = {
    "run_at_utc": datetime.now(UTC).isoformat(),
    "git_revision": git_revision(),
    "notebook_status": "draft-not-run",
}
print("Repository:", REPOSITORY_ROOT)
print("Git revision:", RUN_CONTEXT["git_revision"])
print("Safety: do not print secrets, raw provider payloads, identifiers, or model artifacts.")


<a id="step-1"></a>
## Step 1 — Define candidate features

### What this notebook does

For each proposed feature, record source, transformation, freshness, unknown/imputation owner, and use permission: online, offline, both, or neither.

### Safe success condition

The result is an explicit, sanitised observation or a stated blocker. It is never an implicit contract approval, production integration, or model promotion.


In [ ]:
from pathlib import Path

REQUIRED_REVIEW_INPUTS = [
  "docs/proposals/canonical-transaction-contract.proposed.md",
  "docs/proposals/legacy-scenario-characterisation.md"
]
missing = [path for path in REQUIRED_REVIEW_INPUTS if not (REPOSITORY_ROOT / path).exists()]
if missing:
    print("GATED — this notebook has not run because required review inputs are absent:")
    for path in missing:
        print(f"- {path}")
    print("Do not substitute fabricated inputs. Record the blocker in the matching experiment record.")
else:
    print("Required review inputs are present. Continue only after confirming their approval status.")


<a id="step-2"></a>
## Step 2 — Compare source availability

### What this notebook does

Compare Plaid, legacy A–F, S01–S08, replay, and corpus schema metadata. Do not create corpus values.

### Safe success condition

The result is an explicit, sanitised observation or a stated blocker. It is never an implicit contract approval, production integration, or model promotion.


<a id="step-3"></a>
## Step 3 — Check point-in-time eligibility

### What this notebook does

Treat absent information as unavailable; never turn it into zero, false, or a fabricated value.

### Safe success condition

The result is an explicit, sanitised observation or a stated blocker. It is never an implicit contract approval, production integration, or model promotion.


<a id="step-4"></a>
## Step 4 — Publish exclusions

### What this notebook does

Keep the matrix proposed until the P0 review accepts it.

### Safe success condition

The result is an explicit, sanitised observation or a stated blocker. It is never an implicit contract approval, production integration, or model promotion.


In [ ]:
from hashlib import sha256

report = {
    "notebook": "04-feature-availability-matrix",
    "status": "proposal-prepared-review-pending",
    "run_context": RUN_CONTEXT,
    "findings": ["A proposed availability matrix is available at docs/proposals/feature-availability.proposed.json."],
    "limitations": ["No feature is approved for online scoring; point-in-time parity is not proven."],
    "decision_recommendation": "proposed — no approval or promotion is implied",
}

report_bytes = json.dumps(report, sort_keys=True, indent=2).encode("utf-8")
print("Sanitised report template:", REPOSITORY_ROOT / "docs/proposals/feature-availability.proposed.json")
print("Template digest:", sha256(report_bytes).hexdigest())
print("Do not write the template until it contains only reviewable, sanitised findings.")


<a id="review"></a>
## Findings, limitations, and next gate

- Findings: the proposed matrix records source coverage and explicit exclusions.
- Limitations: no online feature set is approved.
- Recommendation: **proposed** — do not approve a contract, feature, policy, or model from this template.
- Next gate: update `docs/experiments/04-feature-availability-matrix.md` after a run, then request the named review decision.

## Reviewer checklist

- [ ] Outputs are cleared and contain no secrets, raw provider data, PII, identifiers, or hidden reasoning.
- [ ] Every result is labelled observed, unsupported, indeterminate, unavailable, or proposed as appropriate.
- [ ] The matching experiment record contains revision, inputs, findings, limitations, and artifact digest.
- [ ] No runtime contract, threshold, model promotion, or payment action was inferred.

